# 04 — Weekly Team Strength

This notebook combines the frozen preseason team strength ratings with the new in season evidence created in `03_Inseason_Feature_Engineering.ipynb`.

The original preseason model remains unchanged.

The weekly model follows a simple principle:

> Preseason team strength is the prior. Completed regular season games provide new evidence.

For each target week, this notebook:

- Loads the original `2026_team_strength.parquet`
- Preserves every preseason rating and component
- Loads the target week's in season feature table
- Applies the shrunk in season net signal to the preseason team strength
- Keeps offense and defense in season signals separate for later use
- Calculates weekly rating movement and updated rankings
- Saves a completely separate weekly team strength file

No original preseason file is overwritten.


In [13]:
from pathlib import Path

import numpy as np
import pandas as pd


## Paths and Weekly Settings

This notebook expects the output from:

`03_Inseason_Feature_Engineering.ipynb`

For Week 2, `TARGET_WEEK = 2`.

In future weeks, the same notebook can be reused by changing only the target week.


In [14]:
PROJECT_ROOT = Path("../..")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
WEEKLY_DATA_DIR = PROCESSED_DIR / "weekly"

SEASON = 2026
TARGET_WEEK = 2

PRESEASON_STRENGTH_PATH = (
    PROCESSED_DIR
    / "2026_team_strength.parquet"
)

INSEASON_FEATURES_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_inseason_features.parquet"
)

OUTPUT_PATH = (
    WEEKLY_DATA_DIR
    / f"week_{TARGET_WEEK:02d}_team_strength.parquet"
)

print("Preseason ratings:", PRESEASON_STRENGTH_PATH)
print("In-season features:", INSEASON_FEATURES_PATH)
print("Weekly output:", OUTPUT_PATH)


Preseason ratings: ..\..\data\processed\2026_team_strength.parquet
In-season features: ..\..\data\processed\weekly\week_02_inseason_features.parquet
Weekly output: ..\..\data\processed\weekly\week_02_team_strength.parquet


# Load Frozen Preseason Team Strength

The preseason team strength file is the final output of the original `05_Team_Strength_Model.ipynb`.

Its `team_strength` value is already expressed in point differential terms, so it can serve directly as the prior for weekly updates.

We immediately rename the important preseason columns so it is impossible to confuse the original ratings with the new weekly ratings.


In [15]:
preseason = pd.read_parquet(
    PRESEASON_STRENGTH_PATH
).copy()

required_preseason_columns = [
    "team",
    "team_strength",
    "baseline_team_strength",
    "personnel_adjustment",
    "personnel_strength",
    "roster_continuity",
    "roster_continuity_adjustment"
]

missing_preseason = [
    column
    for column in required_preseason_columns
    if column not in preseason.columns
]

if missing_preseason:
    raise KeyError(
        "Missing required preseason columns: "
        + ", ".join(missing_preseason)
    )

preseason = preseason.rename(
    columns={
        "strength_rank": "preseason_strength_rank",
        "team_strength": "preseason_team_strength",
        "baseline_team_strength": "preseason_baseline_team_strength",
        "personnel_adjustment": "preseason_personnel_adjustment",
        "personnel_strength": "preseason_personnel_strength",
        "roster_continuity": "preseason_roster_continuity",
        "roster_continuity_adjustment": "preseason_roster_continuity_adjustment"
    }
)

print("Preseason teams:", len(preseason))
print("Unique teams:", preseason["team"].nunique())

display(
    preseason[
        [
            "preseason_strength_rank",
            "team",
            "preseason_team_strength",
            "preseason_baseline_team_strength",
            "preseason_personnel_adjustment",
            "preseason_roster_continuity_adjustment"
        ]
    ].sort_values(
        "preseason_strength_rank"
    )
)


Preseason teams: 32
Unique teams: 32


,preseason_strength_rank,team,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_roster_continuity_adjustment
0,1,BUF,4.816514,4.125764,0.934506,0.260091
1,2,LA,4.762452,3.257133,2.100811,0.489665
2,3,SEA,4.363421,3.515976,0.635683,0.932073
3,4,DET,4.018565,3.374644,1.041923,0.037533
4,5,DEN,3.569507,2.355154,0.979753,1.253003
5,6,BAL,3.074111,2.710864,0.636254,-0.037012
6,7,HOU,2.810133,2.193179,1.173833,-0.174691
7,8,GB,2.536317,1.718369,0.882628,0.576742
8,9,PHI,2.451919,2.324982,0.264690,-0.063756
9,10,SF,2.366788,1.805709,0.692939,0.290631


# Load In-Season Features

These features contain only games completed before the target week.

The key values used here are:

- `shrunk_offense_signal`
- `shrunk_defense_signal`
- `shrunk_net_signal`
- `inseason_reliability`

The shrinkage already occurred in notebook `03`.

That means this notebook does not apply a second arbitrary Week 1 weighting. The weekly movement is simply the preseason prior plus the already shrunk new information.


In [16]:
inseason = pd.read_parquet(
    INSEASON_FEATURES_PATH
).copy()

required_inseason_columns = [
    "team",
    "games_played",
    "inseason_reliability",
    "shrunk_offense_signal",
    "shrunk_defense_signal",
    "shrunk_net_signal",
    "inseason_offense_z",
    "inseason_defense_z",
    "inseason_net_z"
]

missing_inseason = [
    column
    for column in required_inseason_columns
    if column not in inseason.columns
]

if missing_inseason:
    raise KeyError(
        "Missing required in-season columns: "
        + ", ".join(missing_inseason)
    )

print("In-season teams:", len(inseason))
print("Unique teams:", inseason["team"].nunique())

display(
    inseason[
        required_inseason_columns
    ].sort_values(
        "shrunk_net_signal",
        ascending=False
    )
)


In-season teams: 32
Unique teams: 32


,team,games_played,inseason_reliability,shrunk_offense_signal,shrunk_defense_signal,shrunk_net_signal,inseason_offense_z,inseason_defense_z,inseason_net_z
14,JAX,1,0.166667,0.773438,1.226562,2.000000,0.827613,1.312477,1.694147
5,CHI,1,0.166667,2.856771,-1.023438,1.833333,3.056873,-1.095124,1.552968
15,KC,1,0.166667,0.523438,1.226562,1.750000,0.560102,1.312477,1.482379
28,SF,1,0.166667,0.190104,1.476562,1.666667,0.203420,1.579988,1.411789
2,BAL,1,0.166667,1.356771,0.143229,1.500000,1.451806,0.153262,1.270610
20,MIN,1,0.166667,1.190104,0.226562,1.416667,1.273465,0.242432,1.200021
18,LV,1,0.166667,0.190104,0.976562,1.166667,0.203420,1.044966,0.988252
24,NYJ,1,0.166667,-0.143229,1.226562,1.083333,-0.153262,1.312477,0.917663
0,ARI,1,0.166667,0.106771,0.893229,1.000000,0.114250,0.955795,0.847073
23,NYG,1,0.166667,0.273438,0.393229,0.666667,0.292590,0.420773,0.564716


# Merge Preseason Prior with In-Season Evidence

The merge must retain all 32 preseason teams.

If the previous week is incomplete, a team could theoretically be missing from the in-season table. Those teams receive zero in-season adjustment rather than being dropped.

For the final locked Week 2 run, all 32 teams should have one completed game and therefore appear in the feature table.


In [17]:
weekly_strength = preseason.merge(
    inseason,
    on="team",
    how="left",
    validate="one_to_one"
)

inseason_fill_columns = [
    "games_played",
    "inseason_reliability",
    "shrunk_offense_signal",
    "shrunk_defense_signal",
    "shrunk_net_signal",
    "inseason_offense_z",
    "inseason_defense_z",
    "inseason_net_z"
]

weekly_strength[inseason_fill_columns] = (
    weekly_strength[inseason_fill_columns]
    .fillna(0.0)
)

print("Merged teams:", len(weekly_strength))
print("Unique teams:", weekly_strength["team"].nunique())


Merged teams: 32
Unique teams: 32


# Weekly Rating Update

The preseason `team_strength` rating is already measured in expected point differential space.

The in-season `shrunk_net_signal` is also measured in points.

Therefore the cleanest first weekly update is:

`weekly_team_strength = preseason_team_strength + shrunk_net_signal`

This has several advantages:

- no sportsbook information enters the rating
- no preseason file is altered
- the units remain interpretable
- the update naturally becomes larger only as in season reliability grows
- offense and defense remain separately available for the weekly scoring model

After Week 1, notebook `03` gives only 1/6 reliability to the observed information, so the movement remains intentionally conservative.


In [18]:
weekly_strength["team_strength_change"] = (
    weekly_strength["shrunk_net_signal"]
)

weekly_strength["weekly_team_strength"] = (
    weekly_strength["preseason_team_strength"]
    + weekly_strength["team_strength_change"]
)

weekly_strength["weekly_offense_adjustment"] = (
    weekly_strength["shrunk_offense_signal"]
)

weekly_strength["weekly_defense_adjustment"] = (
    weekly_strength["shrunk_defense_signal"]
)

display(
    weekly_strength[
        [
            "team",
            "preseason_team_strength",
            "weekly_offense_adjustment",
            "weekly_defense_adjustment",
            "team_strength_change",
            "weekly_team_strength"
        ]
    ].sort_values(
        "weekly_team_strength",
        ascending=False
    )
)


,team,preseason_team_strength,weekly_offense_adjustment,weekly_defense_adjustment,team_strength_change,weekly_team_strength
0,BUF,4.816514,0.940104,-0.523438,0.416667,5.233180
2,SEA,4.363421,-0.976562,1.226562,0.250000,4.613421
5,BAL,3.074111,1.356771,0.143229,1.500000,4.574111
3,DET,4.018565,0.523438,-0.440104,0.083333,4.101898
9,SF,2.366788,0.190104,1.476562,1.666667,4.033455
11,JAX,1.713136,0.773438,1.226562,2.000000,3.713136
1,LA,4.762452,-1.476562,-0.190104,-1.666667,3.095785
12,KC,1.302244,0.523438,1.226562,1.750000,3.052244
8,PHI,2.451919,-0.059896,0.226562,0.166667,2.618585
6,HOU,2.810133,0.523438,-0.940104,-0.416667,2.393466


# Updated Weekly Rankings

The weekly rank is based on `weekly_team_strength`.

We also calculate how many ranking positions each team moved relative to the frozen preseason ranking.

A positive `rank_change` means the team moved **up** the rankings.


In [19]:
weekly_strength = weekly_strength.sort_values(
    [
        "weekly_team_strength",
        "preseason_team_strength"
    ],
    ascending=[False, False]
).reset_index(drop=True)

weekly_strength["weekly_strength_rank"] = (
    np.arange(1, len(weekly_strength) + 1)
)

weekly_strength["rank_change"] = (
    weekly_strength["preseason_strength_rank"]
    - weekly_strength["weekly_strength_rank"]
)

display(
    weekly_strength[
        [
            "weekly_strength_rank",
            "team",
            "weekly_team_strength",
            "preseason_strength_rank",
            "preseason_team_strength",
            "team_strength_change",
            "rank_change"
        ]
    ]
)


,weekly_strength_rank,team,weekly_team_strength,preseason_strength_rank,preseason_team_strength,team_strength_change,rank_change
0,1,BUF,5.233180,1,4.816514,0.416667,0
1,2,SEA,4.613421,3,4.363421,0.250000,1
2,3,BAL,4.574111,6,3.074111,1.500000,3
3,4,DET,4.101898,4,4.018565,0.083333,0
4,5,SF,4.033455,10,2.366788,1.666667,5
5,6,JAX,3.713136,12,1.713136,2.000000,6
6,7,LA,3.095785,2,4.762452,-1.666667,-5
7,8,KC,3.052244,13,1.302244,1.750000,5
8,9,PHI,2.618585,9,2.451919,0.166667,0
9,10,HOU,2.393466,7,2.810133,-0.416667,-3


# Biggest Weekly Movers

This table is diagnostic only.

It lets us see whether Week 1 is moving teams by a sensible amount before those ratings are used for Week 2 predictions.


In [20]:
biggest_risers = (
    weekly_strength[
        [
            "team",
            "preseason_team_strength",
            "team_strength_change",
            "weekly_team_strength",
            "preseason_strength_rank",
            "weekly_strength_rank",
            "rank_change"
        ]
    ]
    .sort_values(
        "team_strength_change",
        ascending=False
    )
    .head(10)
)

biggest_fallers = (
    weekly_strength[
        [
            "team",
            "preseason_team_strength",
            "team_strength_change",
            "weekly_team_strength",
            "preseason_strength_rank",
            "weekly_strength_rank",
            "rank_change"
        ]
    ]
    .sort_values(
        "team_strength_change",
        ascending=True
    )
    .head(10)
)

print("BIGGEST RISERS")
display(biggest_risers)

print()
print("BIGGEST FALLERS")
display(biggest_fallers)


BIGGEST RISERS


,team,preseason_team_strength,team_strength_change,weekly_team_strength,preseason_strength_rank,weekly_strength_rank,rank_change
5,JAX,1.713136,2.000000,3.713136,12,6,6
12,CHI,-0.051934,1.833333,1.781399,19,13,6
7,KC,1.302244,1.750000,3.052244,13,8,5
4,SF,2.366788,1.666667,4.033455,10,5,5
2,BAL,3.074111,1.500000,4.574111,6,3,3
10,MIN,0.475040,1.416667,1.891707,18,11,7
27,LV,-5.693342,1.166667,-4.526676,30,28,2
28,NYJ,-5.933726,1.083333,-4.850393,31,29,2
24,ARI,-3.678016,1.000000,-2.678016,27,25,2
25,NYG,-3.455862,0.666667,-2.789195,26,26,0



BIGGEST FALLERS


,team,preseason_team_strength,team_strength_change,weekly_team_strength,preseason_strength_rank,weekly_strength_rank,rank_change
29,CLE,-4.150395,-2.000000,-6.150395,28,30,-2
30,CAR,-4.533242,-1.833333,-6.366575,29,31,-2
11,DEN,3.569507,-1.750000,1.819507,5,12,-7
6,LA,4.762452,-1.666667,3.095785,2,7,-5
19,IND,0.532226,-1.500000,-0.967774,17,20,-3
15,GB,2.536317,-1.416667,1.119650,8,16,-8
26,MIA,-2.443254,-1.166667,-3.609921,25,27,-2
31,TEN,-6.747344,-1.083333,-7.830677,32,32,0
17,LAC,0.910681,-1.000000,-0.089319,14,18,-4
20,DAL,-0.386396,-0.666667,-1.053063,20,21,-1


# Update Diagnostics

These diagnostics are meant to catch an overly aggressive weekly update.

After Week 1, rating movement should be noticeably smaller than the underlying one game point differential because the in season features were heavily shrunk in notebook `03`.


In [21]:
movement = weekly_strength["team_strength_change"].abs()

print("WEEKLY TEAM-STRENGTH MOVEMENT")
print(
    weekly_strength["team_strength_change"]
    .describe()
    .round(3)
)

print()
print(
    "Mean absolute movement:",
    round(movement.mean(), 3)
)

print(
    "Maximum absolute movement:",
    round(movement.max(), 3)
)

print(
    "Team with largest movement:",
    weekly_strength.loc[
        movement.idxmax(),
        "team"
    ]
)

print()
print("RATING DISTRIBUTIONS")

distribution_check = pd.DataFrame(
    {
        "metric": [
            "Preseason Team Strength",
            "Weekly Team Strength"
        ],
        "mean": [
            weekly_strength["preseason_team_strength"].mean(),
            weekly_strength["weekly_team_strength"].mean()
        ],
        "std": [
            weekly_strength["preseason_team_strength"].std(),
            weekly_strength["weekly_team_strength"].std()
        ],
        "min": [
            weekly_strength["preseason_team_strength"].min(),
            weekly_strength["weekly_team_strength"].min()
        ],
        "max": [
            weekly_strength["preseason_team_strength"].max(),
            weekly_strength["weekly_team_strength"].max()
        ]
    }
)

display(distribution_check.round(3))


WEEKLY TEAM-STRENGTH MOVEMENT
count    32.000
mean      0.000
std       1.199
min      -2.000
25%      -1.021
50%       0.000
75%       1.021
max       2.000
Name: team_strength_change, dtype: float64

Mean absolute movement: 1.005
Maximum absolute movement: 2.0
Team with largest movement: JAX

RATING DISTRIBUTIONS


,metric,mean,std,min,max
0,Preseason Team Strength,0.002,3.266,-6.747,4.817
1,Weekly Team Strength,0.002,3.539,-7.831,5.233


# Sanity Checks

These checks protect the separation between the preseason model and weekly model.

The weekly file should:

- contain exactly 32 unique teams
- preserve every preseason rating
- contain no missing weekly strength values
- have `team_strength_change` exactly equal to the shrunk in season net signal
- never overwrite the original preseason parquet file


In [22]:
assert len(weekly_strength) == 32, (
    "Expected 32 teams in weekly strength table."
)

assert weekly_strength["team"].nunique() == 32, (
    "Expected 32 unique teams."
)

assert weekly_strength[
    "preseason_team_strength"
].notna().all(), (
    "Missing frozen preseason ratings."
)

assert weekly_strength[
    "weekly_team_strength"
].notna().all(), (
    "Missing weekly team-strength ratings."
)

assert np.allclose(
    weekly_strength["team_strength_change"],
    weekly_strength["shrunk_net_signal"]
), (
    "Weekly movement no longer matches the intended "
    "shrunk in-season signal."
)

assert np.allclose(
    weekly_strength["weekly_team_strength"],
    (
        weekly_strength["preseason_team_strength"]
        + weekly_strength["team_strength_change"]
    )
), (
    "Weekly team-strength formula is inconsistent."
)

print("All weekly team strength sanity checks passed.")


All weekly team strength sanity checks passed.


# Final Weekly Team-Strength Table

The saved file keeps both the original preseason information and the new weekly information.

This is deliberate: every weekly prediction can later be traced back to exactly how far the team's rating moved from its preseason prior.


In [23]:
final_columns = [
    "weekly_strength_rank",
    "team",
    "weekly_team_strength",
    "team_strength_change",
    "rank_change",
    "preseason_strength_rank",
    "preseason_team_strength",
    "preseason_baseline_team_strength",
    "preseason_personnel_adjustment",
    "preseason_personnel_strength",
    "preseason_roster_continuity",
    "preseason_roster_continuity_adjustment",
    "games_played",
    "inseason_reliability",
    "weekly_offense_adjustment",
    "weekly_defense_adjustment",
    "shrunk_net_signal",
    "inseason_offense_z",
    "inseason_defense_z",
    "inseason_net_z"
]

weekly_team_strength = weekly_strength[
    final_columns
].copy()

display(weekly_team_strength)


,weekly_strength_rank,team,weekly_team_strength,team_strength_change,rank_change,preseason_strength_rank,preseason_team_strength,preseason_baseline_team_strength,preseason_personnel_adjustment,preseason_personnel_strength,preseason_roster_continuity,preseason_roster_continuity_adjustment,games_played,inseason_reliability,weekly_offense_adjustment,weekly_defense_adjustment,shrunk_net_signal,inseason_offense_z,inseason_defense_z,inseason_net_z
0,1,BUF,5.233180,0.416667,0,1,4.816514,4.125764,0.934506,0.623004,0.573034,0.260091,1,0.166667,0.940104,-0.523438,0.416667,1.005954,-0.560102,0.352947
1,2,SEA,4.613421,0.250000,1,3,4.363421,3.515976,0.635683,0.423788,0.677419,0.932073,1,0.166667,-0.976562,1.226562,0.250000,-1.044966,1.312477,0.211768
2,3,BAL,4.574111,1.500000,3,6,3.074111,2.710864,0.636254,0.424170,0.526882,-0.037012,1,0.166667,1.356771,0.143229,1.500000,1.451806,0.153262,1.270610
3,4,DET,4.101898,0.083333,0,4,4.018565,3.374644,1.041923,0.694616,0.538462,0.037533,1,0.166667,0.523438,-0.440104,0.083333,0.560102,-0.470931,0.070589
4,5,SF,4.033455,1.666667,5,10,2.366788,1.805709,0.692939,0.461959,0.577778,0.290631,1,0.166667,0.190104,1.476562,1.666667,0.203420,1.579988,1.411789
5,6,JAX,3.713136,2.000000,6,12,1.713136,1.746953,-0.223729,-0.149153,0.563830,0.200841,1,0.166667,0.773438,1.226562,2.000000,0.827613,1.312477,1.694147
6,7,LA,3.095785,-1.666667,-5,2,4.762452,3.257133,2.100811,1.400540,0.608696,0.489665,1,0.166667,-1.476562,-0.190104,-1.666667,-1.579988,-0.203420,-1.411789
7,8,KC,3.052244,1.750000,5,13,1.302244,1.484935,-0.187743,-0.125162,0.510870,-0.140090,1,0.166667,0.523438,1.226562,1.750000,0.560102,1.312477,1.482379
8,9,PHI,2.618585,0.166667,0,9,2.451919,2.324982,0.264690,0.176460,0.522727,-0.063756,1,0.166667,-0.059896,0.226562,0.166667,-0.064091,0.242432,0.141179
9,10,HOU,2.393466,-0.416667,-3,7,2.810133,2.193179,1.173833,0.782555,0.505495,-0.174691,1,0.166667,0.523438,-0.940104,-0.416667,0.560102,-1.005954,-0.352947


# Save Weekly Team Strength

This writes only to:

`data/processed/weekly/`

The original:

`data/processed/2026_team_strength.parquet`

remains untouched.


In [24]:
OUTPUT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)

weekly_team_strength.to_parquet(
    OUTPUT_PATH,
    index=False
)

print("Saved:", OUTPUT_PATH)
print()
print(
    "Original preseason file remains:",
    PRESEASON_STRENGTH_PATH
)


Saved: ..\..\data\processed\weekly\week_02_team_strength.parquet

Original preseason file remains: ..\..\data\processed\2026_team_strength.parquet


# What This Means for Week 2

At the end of this notebook, every team has:

- its frozen preseason rating
- its Week 1 offensive in season adjustment
- its Week 1 defensive in season adjustment
- its net Week 1 rating movement
- an updated Week 2 team-strength rating
- an updated Week 2 rank

The next notebook, `05_Weekly_Game_Predictions.ipynb`, will use `weekly_team_strength` instead of the frozen preseason `team_strength` when calculating Week 2 expected margins.

The original preseason game projections will remain available as a separate baseline so we can compare:

**Preseason Week 2 projection → Updated Week 2 projection → Actual result**
